In [1]:
import os
import csv
from gift_eval.data import Dataset
from gluonts.ev.metrics import (MAE,MAPE,MASE,MSE,MSIS,ND,NRMSE,RMSE,SMAPE,MeanWeightedSumQuantileLoss,)
from gluonts.model import evaluate_model
from gluonts.time_feature import get_seasonality
import json
import numpy as np
import pandas as pd
import random
import torch
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Experiment configurations
short_datasets = "m4_yearly m4_quarterly m4_monthly m4_weekly m4_daily m4_hourly electricity/15T electricity/H electricity/D electricity/W solar/10T solar/H solar/D solar/W hospital covid_deaths us_births/D us_births/M us_births/W saugeenday/D saugeenday/M saugeenday/W temperature_rain_with_missing kdd_cup_2018_with_missing/H kdd_cup_2018_with_missing/D car_parts_with_missing restaurant hierarchical_sales/D hierarchical_sales/W LOOP_SEATTLE/5T LOOP_SEATTLE/H LOOP_SEATTLE/D SZ_TAXI/15T SZ_TAXI/H M_DENSE/H M_DENSE/D ett1/15T ett1/H ett1/D ett1/W ett2/15T ett2/H ett2/D ett2/W jena_weather/10T jena_weather/H jena_weather/D bitbrains_fast_storage/5T bitbrains_fast_storage/H bitbrains_rnd/5T bitbrains_rnd/H bizitobs_application bizitobs_service bizitobs_l2c/5T bizitobs_l2c/H"
med_long_datasets = "electricity/15T electricity/H solar/10T solar/H kdd_cup_2018_with_missing/H LOOP_SEATTLE/5T LOOP_SEATTLE/H SZ_TAXI/15T M_DENSE/H ett1/15T ett1/H ett2/15T ett2/H jena_weather/10T jena_weather/H bitbrains_fast_storage/5T bitbrains_rnd/5T bizitobs_application bizitobs_service bizitobs_l2c/5T bizitobs_l2c/H"
dataset_properties = 'dataset_properties.json'

In [13]:
# Auxiliary functions
def set_seed(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
from zeus.modeling_zeus import ZeusForPrediction
import torch
import numpy as np
from tqdm.auto import tqdm
from gluonts.itertools import batcher
from gluonts.model.forecast import QuantileForecast

class ZeusPredictor:
    def __init__(
        self,
        prediction_length: int,
        model_path: str,
        device: str = "cuda",
        batch_size: int = 2048,
        context_length: int = 4096
    ):
        self.device = device
        self.context_length = context_length

        self.model = ZeusForPrediction.from_pretrained(
            model_path,
            attn_implementation="flash_attention_2",
            device_map=device
        )

        self.prediction_length = prediction_length
        self.batch_size = batch_size
        print(f"Context length: {self.context_length}, Prediction length: {self.prediction_length}")

    def predict(
        self,
        test_data_input,
        batch_size: int = 160,
    ):
        
        forecast_outputs = []
        with torch.no_grad():
            for batch in tqdm(batcher(test_data_input, batch_size=batch_size)):
                inputs = [entry["target"] for entry in batch] # list of [N, L]
                _, quantiles = self.model.predict(
                        inputs,
                        prediction_length=self.prediction_length,
                        normalize=True,
                        max_pred_len=min(self.context_length + self.prediction_length, 4096)
                    )
                # print(quantiles)
                # [B, N, L, Q] -> [B, Q, L, N]
                if quantiles.ndim == 4:
                    quantiles = quantiles.permute(0, 3, 2, 1)
                else:
                    # [B, L, Q] -> [B, Q, L]
                    quantiles = np.transpose(quantiles, (0, 2, 1))
                # print(quantiles.shape)
                
                # forecast_outputs [batch, quantiles, seq_len, variates]
                if quantiles.shape[-1] == 1:
                    quantiles = quantiles.squeeze(-1) # squeeze variate to avoid error in eval due to broadcasting
                assert quantiles.shape[1] == 9
                assert quantiles.shape[2] == self.prediction_length
                # quantiles = quantiles.detach().cpu().numpy()
                forecast_outputs.append(quantiles)
        forecast_outputs = np.concatenate(forecast_outputs, axis=0)

        # Convert forecast samples into gluonts Forecast objects
        forecasts = []
        for item, ts in zip(forecast_outputs, test_data_input):
            forecast_start_date = ts["start"] + len(ts["target"])
            forecasts.append(
                QuantileForecast(
                    forecast_arrays=item,
                    forecast_keys=list(map(str, self.config.quantiles)),
                    start_date=forecast_start_date,
                )
            )

        return forecasts

In [7]:
# Path configurations
model_dir = './checkpoints'
config_dir = './configs'
out_dir = './results/Zeus'
model_name = 'Zeus'
model_path = "zeus"

# Auxiliary configurations
seed = 42
device = 'cuda'
batch_size = 8

In [ ]:
import logging


class WarningFilter(logging.Filter):
    def __init__(self, text_to_filter):
        super().__init__()
        self.text_to_filter = text_to_filter

    def filter(self, record):
        return self.text_to_filter not in record.getMessage()


gts_logger = logging.getLogger("gluonts.model.forecast")
gts_logger.addFilter(
    WarningFilter("The mean prediction is not stored in the forecast data")
)

In [ ]:
all_datasets = list(set(short_datasets.split() + med_long_datasets.split()))
all_datasets.sort(key=str.lower)

In [ ]:
dataset_properties_map = json.load(open("dataset_properties.json"))

# Instantiate the metrics
metrics = [
    MSE(forecast_type="mean"),
    MSE(forecast_type=0.5),
    MAE(),
    MASE(),
    MAPE(),
    SMAPE(),
    MSIS(),
    RMSE(),
    NRMSE(),
    ND(),
    MeanWeightedSumQuantileLoss(
        quantile_levels=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
    ),
]

pretty_names = {
    "saugeenday": "saugeen",
    "temperature_rain_with_missing": "temperature_rain",
    "kdd_cup_2018_with_missing": "kdd_cup_2018",
    "car_parts_with_missing": "car_parts",
}

csv_file_path = os.path.join(out_dir, "all_results.csv")

completed_datasets = set()
# 1. Check if the results file exists and read the completed datasets to allow resuming
if os.path.exists(csv_file_path):
    print(f"'{csv_file_path}' exists. Reading completed datasets...")
    with open(csv_file_path, "r", newline="") as csvfile:
        reader = csv.reader(csvfile)
        next(reader)
        for row in reader:
            if row:
                completed_datasets.add(row[0])
    print(f"Found {len(completed_datasets)} completed datasets.")

# 2. If the file doesn't exist, create it and write the header
else:
    with open(csv_file_path, "w", newline="") as csvfile:
        writer = csv.writer(csvfile)

        # Write the header
        writer.writerow(
            [
                "dataset",
                "model",
                "eval_metrics/MSE[mean]",
                "eval_metrics/MSE[0.5]",
                "eval_metrics/MAE[0.5]",
                "eval_metrics/MASE[0.5]",
                "eval_metrics/MAPE[0.5]",
                "eval_metrics/sMAPE[0.5]",
                "eval_metrics/MSIS",
                "eval_metrics/RMSE[mean]",
                "eval_metrics/NRMSE[mean]",
                "eval_metrics/ND[0.5]",
                "eval_metrics/mean_weighted_sum_quantile_loss",
                "domain",
                "num_variates"
            ]
        )

for ds_num, ds_name in enumerate(all_datasets):
    ds_key = ds_name.split("/")[0]
    print(f"Processing dataset: {ds_name} ({ds_num + 1} of {len(all_datasets)})")
    terms = ["long", "medium", "short"]
    for term in terms:
        if (
            term == "medium" or term == "long"
        ) and ds_name not in med_long_datasets.split():
            continue

        if "/" in ds_name:
            ds_key = ds_name.split("/")[0]
            ds_freq = ds_name.split("/")[1]
            ds_key = ds_key.lower()
            ds_key = pretty_names.get(ds_key, ds_key)
        else:
            ds_key = ds_name.lower()
            ds_key = pretty_names.get(ds_key, ds_key)
            ds_freq = dataset_properties_map[ds_key]["frequency"]
        ds_config = f"{ds_key}/{ds_freq}/{term}"

        if ds_config in completed_datasets:
            print(f"Skipping already completed dataset: {ds_config}")
            continue

        # Initialize the dataset
        to_univariate = (
            False
            if Dataset(name=ds_name, term=term, to_univariate=False).target_dim == 1
            else True
        )
        # to_univariate = False
        dataset = Dataset(name=ds_name, term=term, to_univariate=to_univariate)
        season_length = get_seasonality(dataset.freq)
        print(f"Dataset size: {len(dataset.test_data)}")

        if term == "short":
            context_list = [128, 512, 1024, 2048, 4096]
        else:
            context_list = [4096, 3072, 2048, 1024, 512]
        
        best_res = None
        best_context_length = None
        for context_length in context_list:
            predictor = ZeusPredictor(
                model_path=model_path,
                prediction_length=dataset.prediction_length,
                context_length=context_length,
                batch_size=batch_size,
                device=device
            )
            # Measure the time taken for evaluation
            res = evaluate_model(
                    predictor,
                    test_data=dataset.test_data,
                    metrics=metrics,
                    batch_size=batch_size,
                    axis=None,
                    mask_invalid_label=True,
                    allow_nan_forecast=False,
                    seasonality=season_length,
                )

            # Update the best results if the current MASE is lower
            if best_res is None or res["MASE[0.5]"][0] < best_res["MASE[0.5]"][0]:
                best_res = res
                best_context_length = context_length
        
        # Append the results to the CSV file
        with open(csv_file_path, "a", newline="") as csvfile:
            writer = csv.writer(csvfile)
            writer.writerow(
                [
                    ds_config,
                    model_name,
                    best_res["MSE[mean]"][0],
                    best_res["MSE[0.5]"][0],
                    best_res["MAE[0.5]"][0],
                    best_res["MASE[0.5]"][0],
                    best_res["MAPE[0.5]"][0],
                    best_res["sMAPE[0.5]"][0],
                    best_res["MSIS"][0],
                    best_res["RMSE[mean]"][0],
                    best_res["NRMSE[mean]"][0],
                    best_res["ND[0.5]"][0],
                    best_res["mean_weighted_sum_quantile_loss"][0],
                    dataset_properties_map[ds_key]["domain"],
                    dataset_properties_map[ds_key]["num_variates"]
                ]
            )

        print(f"Results for {ds_name} have been written to {csv_file_path}")

In [4]:
res = pd.read_csv(f"../results/Zeus/all_results.csv")

In [8]:
# Print Results
seasonal_naive = pd.read_csv(f"../results/seasonal_naive/all_results.csv").sort_values('dataset')
dataset = seasonal_naive['dataset'].to_list()
seasonal_naive_mase = seasonal_naive[f'eval_metrics/MASE[0.5]'].to_numpy()
seasonal_naive_crps = seasonal_naive[f'eval_metrics/mean_weighted_sum_quantile_loss'].to_numpy()
df = res.sort_values(by="dataset")
df['normalized MASE'] = np.zeros(len(df))
df['normalized CRPS'] = np.zeros(len(df))
df['freq'] = np.zeros(len(df))
df['len'] = np.zeros(len(df))
for ds in df['dataset']:
    idx = dataset.index(ds)
    _, f, l = ds.split('/')
    df.loc[df['dataset'] ==ds, 'freq'] = f
    df.loc[df['dataset'] ==ds, 'len'] = l
    df.loc[df['dataset'] ==ds, 'normalized MASE'] = df.loc[df['dataset'] ==ds, f'eval_metrics/MASE[0.5]'].values / seasonal_naive_mase[idx]
    df.loc[df['dataset'] ==ds, 'normalized CRPS'] = df.loc[df['dataset'] ==ds, f'eval_metrics/mean_weighted_sum_quantile_loss'].values / seasonal_naive_crps[idx]


df = df.sort_values(by=['dataset'])
def geo_mean(iterable):
    a = np.array(iterable)
    return a.prod()**(1.0/len(a))

mase = geo_mean(df['normalized MASE'].to_numpy())
crps = geo_mean(df['normalized CRPS'].to_numpy())

print(f'Final GIFT-Eval Performance of {model_name}:\nMASE = {mase}, CRPS = {crps}')

Final GIFT-Eval Performance of Zeus:
MASE = 0.6930683106459659, CRPS = 0.4803375635255624
